# Notebook 03: Deep Learning U-Net Training & Segmentation

Here we train **Model B**, a dual-head lightweight U-Net on MaVeCoDD corrosion segmentation. We optimize a combined Binary Cross-Entropy + Soft Dice Loss to overcome foreground-background class imbalance.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
from src.utils.config import load_config
from src.utils.reproducibility import get_device
from src.data.split_dataset import load_split_manifest
from src.models.cnn import HullUNet
from src.models.train import train_unet

config = load_config()
dev = get_device('auto')
print('Using device:', dev)
splits = load_split_manifest('../data/processed/split_manifest_mavecodd.json')


## 1. Model Architecture & Parameters


In [ ]:
unet = HullUNet(in_channels=3, out_channels=1, base_channels=24).to(dev)
total_params = sum(p.numel() for p in unet.parameters())
print(f'Total trainable parameters: {total_params:,}')


## 2. Train U-Net on Training Split


In [ ]:
config['cnn_model']['epochs'] = 5
results = train_unet(splits['train'], splits['val'], config=config, device=dev)
print('Training complete! Best val loss:', results['best_val_loss'])
